In [ ]:
import csv
import numpy as np
import pandas as pd
import plotly.express as px

# File paths
#expense_file = r'C:\Users\Bruger 1\Desktop\csv\expense.csv'
#income_file = r'C:\Users\Bruger 1\Desktop\csv\income.csv'
#users_file = r'C:\Users\Bruger 1\Desktop\csv\user (1).csv'

# General CSV Writing Function
def write_to_csv(file_path, fieldnames, data, overwrite=False):
    try:
        mode = 'w' if overwrite else 'a'
        with open(file_path, mode=mode, newline='') as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            if overwrite:  # Write headers only when overwriting
                writer.writeheader()
            for row in data:
                writer.writerow(row)
            print(f"Data saved to {file_path}")
    except Exception as e:
        print(f"Error writing data: {e}")

# Reading Functions
def read_expenses():
    expenses = []
    try:
        with open(expense_file, mode='r', newline='') as file:
            reader = csv.DictReader(file)
            for row in reader:
                expenses.append(row)
    except Exception as e:
        print(f"Error reading expense data: {e}")
    return expenses

def read_incomes():
    incomes = []
    try:
        with open(income_file, mode='r', newline='') as file:
            reader = csv.DictReader(file)
            for row in reader:
                incomes.append(row)
    except Exception as e:
        print(f"Error reading income data: {e}")
    return incomes

def read_users():
    users = {}
    try:
        with open(users_file, mode='r', newline='') as file:
            reader = csv.DictReader(file)
            for row in reader:
                users[row['UserID']] = row['UserName']
    except Exception as e:
        print(f"Error reading user data: {e}")
    return users

# Write to CSV for income/expense records
def write_expense(expense):
    fieldnames = ['UserID', 'UserName', 'ExpenseID', 'Amount', 'Category', 'BudgetLimit']
    write_to_csv(expense_file, fieldnames, [expense], overwrite=False)

def write_income(income):
    fieldnames = ['UserID', 'UserName', 'IncomeID', 'Amount', 'Source']
    write_to_csv(income_file, fieldnames, [income], overwrite=False)

# Write new user to CSV
def write_user(user):
    fieldnames = ['UserID', 'UserName']
    write_to_csv(users_file, fieldnames, [user], overwrite=False)

# Add a new user
def add_user():
    user_id = input("Enter a unique User ID: ").strip()
    user_name = input("Enter the User Name: ").strip()

    users = read_users()
    if user_id in users:
        print(f"User with ID {user_id} already exists.")
        return

    new_user = {'UserID': user_id, 'UserName': user_name}
    write_user(new_user)

# Delete User from the CSV
def delete_user():
    user_id = input("Enter the User ID to delete: ").strip()
    users = read_users()

    if user_id not in users:
        print(f"No user found with User ID {user_id}.")
        return

    # Remove the user from the dictionary
    del users[user_id]

    # Update the users.csv by writing the updated list of users back to the file
    fieldnames = ['UserID', 'UserName']
    updated_users = [{'UserID': user_id, 'UserName': user_name} for user_id, user_name in users.items()]

    write_to_csv(users_file, fieldnames, updated_users, overwrite=True)

    print(f"User with User ID {user_id} has been deleted.")

# View Users
def view_users():
    users = read_users()
    print("\n--- User List ---")
    for user_id, user_name in users.items():
        print(f"UserID: {user_id}, UserName: {user_name}")

# Function to check if the user is approaching or exceeding their budget
def check_budget_limit(category, amount, user_name):
    # Read all expenses
    expenses = pd.DataFrame(read_expenses())
    
    # Filter expenses by the user and category
    user_expenses = expenses[(expenses['UserName'] == user_name) & (expenses['Category'] == category)]
    
    # Calculate the total expense for this category
    # total_expense = user_expenses['Amount'].astype(float).sum()
    # total_expense += amount  # Add the current expense to the total
    
    # Get the budget limit for this category (assuming all expenses in a category have the same limit)
    budget_limit = int(float(user_expenses[user_expenses['Category'] == category]['BudgetLimit'].iloc[0]))

    # Check if the total expense exceeds or approaches the budget limit
    if amount > budget_limit:
        print(f"Warning: You have exceeded your budget for {category}. Total expense: ${amount:.2f}, Budget limit: ${budget_limit:.2f}")
    elif amount >= 0.9 * budget_limit:
        print(f"Warning: You are approaching your budget limit for {category}. Total expense: ${amount:.2f}, Budget limit: ${budget_limit:.2f}")

# Add Income/Expense Record
def add_record(user_id, user_name):
    record_type = input("Enter record type (Income/Expense): ").strip().lower()

    if record_type not in ['income', 'expense']:
        print("Invalid record type. Please choose either 'Income' or 'Expense'.")
        return

    if record_type == 'expense':
        expense_id = input("Enter the Expense ID: ").strip()
        category = input("Enter the expense category (e.g., Food, Rent): ").strip()
        amount = input("Enter the amount: $").strip()
        try:
            amount = float(amount)  # Ensure it's a valid float
        except ValueError:
            print("Invalid amount. Please enter a numeric value.")
            return
        budget_limit = input("Enter the budget limit for this category: $").strip()
        try:
            budget_limit = float(budget_limit)  # Ensure it's a valid float
        except ValueError:
            print("Invalid budget limit. Please enter a numeric value.")
            return

        # Write the expense to the CSV file
        expense = {'UserID': user_id, 'UserName': user_name, 'ExpenseID': expense_id,
                   'Amount': f'{amount:.2f}', 'Category': category, 'BudgetLimit': f'{budget_limit:.2f}'}
        write_expense(expense)
        print(f"Expense record added: {category} - ${amount} with budget limit ${budget_limit}")

        # Now check if the user is approaching or exceeding their budget for this category
        check_budget_limit(category, amount, user_name)

    elif record_type == 'income':
        income_id = input("Enter the Income ID: ").strip()
        source = input("Enter the income source (e.g., Salary, Freelance): ").strip()
        amount = input("Enter the amount: $").strip()
        try:
            amount = float(amount)  # Ensure it's a valid float
        except ValueError:
            print("Invalid amount. Please enter a numeric value.")
            return

        income = {'UserID': user_id, 'UserName': user_name, 'IncomeID': income_id,
                  'Amount': f'{amount:.2f}', 'Source': source}
        write_income(income)
        print(f"Income record added: {source} - ${amount}")

# View User Records (Income/Expense)
def view_user_records(user_name):
    expenses = read_expenses()
    incomes = read_incomes()

    user_expenses = [exp for exp in expenses if exp['UserName'] == user_name]
    user_incomes = [inc for inc in incomes if inc['UserName'] == user_name]

    print(f"\n--- Records for {user_name} ---")

    # Display expenses and check budget limits
    if user_expenses:
        print("\nExpenses:")
        for exp in user_expenses:
            print(f"ExpenseID: {exp['ExpenseID']}, Category: {exp['Category']}, "
                  f"Amount: ${exp['Amount']}, Budget Limit: ${exp['BudgetLimit']}")
            # Check budget status for each category
            check_budget_limit(exp['Category'], float(exp['Amount']), user_name)
    else:
        print("No expense records found.")

    # Display incomes
    if user_incomes:
        print("\nIncomes:")
        for inc in user_incomes:
            print(f"IncomeID: {inc['IncomeID']}, Source: {inc['Source']}, Amount: ${inc['Amount']}")
    else:
        print("No income records found.")

# Authenticate User by UserID
def authenticate_user():
    user_id = input("Enter your User ID: ").strip()
    users = read_users()
    if user_id in users:
        return users[user_id], user_id
    else:
        print("User not found.")
        return None, None

# Delete Income Record
def delete_income(user_name):
    income_id = input("Enter the Income ID to delete: ").strip()
    incomes = read_incomes()

    # Filter out the income record to be deleted
        # Filter out the income record to be deleted
    incomes = [inc for inc in incomes if not (inc['UserName'] == user_name and inc['IncomeID'] == income_id)]

    # Write the updated list back to the CSV
    write_to_csv(income_file, ['UserID', 'UserName', 'IncomeID', 'Amount', 'Source'], incomes, overwrite=True)
    print(f"Income record with Income ID {income_id} deleted.")

# Delete Expense Record
def delete_expense(user_name):
    expense_id = input("Enter the Expense ID to delete: ").strip()
    expenses = read_expenses()

    # Filter out the expense record to be deleted
    expenses = [exp for exp in expenses if not (exp['UserName'] == user_name and exp['ExpenseID'] == expense_id)]

    # Write the updated list back to the CSV
    write_to_csv(expense_file, ['UserID', 'UserName', 'ExpenseID', 'Amount', 'Category', 'BudgetLimit'], expenses, overwrite=True)
    print(f"Expense record with Expense ID {expense_id} deleted.")

def view_summary(user_name):
    expenses = pd.DataFrame(read_expenses())  # Read expenses
    incomes = pd.DataFrame(read_incomes())    # Read incomes

    # Filter data by user_name
    user_expenses = expenses[expenses['UserName'] == user_name]
    user_incomes = incomes[incomes['UserName'] == user_name]

    # Summarize expenses by Category
    expense_summary = user_expenses.groupby('Category').agg({'Amount': 'sum'}).reset_index()
    expense_summary['Type'] = 'Expense'

    expense_summary_budget = user_expenses.groupby('Category').agg({'BudgetLimit': 'sum'}).reset_index()
    expense_summary_budget['Type'] = 'Expense'

    # Summarize incomes by Source
    income_summary = user_incomes.groupby('Source').agg({'Amount': 'sum'}).reset_index()
    income_summary['Type'] = 'Income'

    # Combine both the income and expense dataframes
    category_summary = pd.concat([expense_summary, income_summary], ignore_index=True)

    # Ensure that 'Category' for expenses and 'Source' for incomes are in the same column for the x-axis
    category_summary['Category/Source'] = category_summary['Category'].fillna(category_summary['Source'])

    
    categories = np.array(category_summary.Category)
    categories = categories[~pd.isnull(categories)]
    budget_limit = np.array(np.array(expense_summary_budget['BudgetLimit']))
    budget_limit = [int(b) for b in budget_limit]

    amount = np.array(expense_summary['Amount'])
    amount = [int(a) for a in amount]
    fig = px.line(x=categories, y=budget_limit, color=px.Constant("Budget Limit"),
             labels=dict(x="Expenses", y="Amount"))
    fig.add_bar(x=categories, y=amount, name="Expenses")
    
    fig.show()

# Main function to run the program
def main():
    while True:
        print("\n--- Budget Management System ---")
        print("1. Add User")
        print("2. Delete User")
        print("3. View Users")
        print("4. Add Income/Expense Record")
        print("5. View User Records")
        print("6. Delete Income Record")
        print("7. Delete Expense Record")
        print("8. View Income and Expense Summary")
        print("9. Exit")

        choice = input("Enter your choice: ").strip()

        if choice == '1':
            add_user()
        elif choice == '2':
            delete_user()
        elif choice == '3':
            view_users()
        elif choice == '4':
            user_name, user_id = authenticate_user()
            if user_name:
                add_record(user_id, user_name)
        elif choice == '5':
            user_name, user_id = authenticate_user()
            if user_name:
                view_user_records(user_name)
        elif choice == '6':
            user_name, user_id = authenticate_user()
            if user_name:
                delete_income(user_name)
        elif choice == '7':
            user_name, user_id = authenticate_user()
            if user_name:
                delete_expense(user_name)
        elif choice == '8':
            user_name, user_id = authenticate_user()
            if user_name:
                view_summary(user_name)
        elif choice == '9':
            print("Exiting...")
            break
        else:
            print("Invalid choice. Please try again.")

if __name__ == "__main__":
    main()




--- Budget Management System ---
1. Add User
2. Delete User
3. View Users
4. Add Income/Expense Record
5. View User Records
6. Delete Income Record
7. Delete Expense Record
8. View Income and Expense Summary
9. Exit
User not found.

--- Budget Management System ---
1. Add User
2. Delete User
3. View Users
4. Add Income/Expense Record
5. View User Records
6. Delete Income Record
7. Delete Expense Record
8. View Income and Expense Summary
9. Exit
Exiting...
